In [ ]:
import pandas as pd
import numpy as np
import os

from scipy.interpolate import interp1d
from concurrent.futures import ThreadPoolExecutor

import warnings
warnings.filterwarnings('ignore')

In [ ]:
VIDEO_PATH = r'data/Videos/'
LANDMARK_PATH = r'landmarks/'
PROCESSED_LANDMARK_PATH = r'landmarks_processed/'
REPORT_PATH = r'VideoReport_updated.csv'

TARGET_FRAMES = 172
TARGET_FPS = 15

os.makedirs(PROCESSED_LANDMARK_PATH, exist_ok=True)

In [ ]:
def downsample(sequence, target_fps, fps):
    stride = max(1, int(round(fps / target_fps)))
    return sequence[::stride]

def interpolate(sequence, target_frames = 150):
    T = sequence.shape[0]

    if T == target_frames:
        return sequence.astype(np.float32)

    old_idx = np.linspace(0, 1, T)
    new_idx = np.linspace(0, 1, target_frames)

    out = np.zeros((target_frames, sequence.shape[1], sequence.shape[2]), dtype=np.float32)

    for landmark in range(sequence.shape[1]):
        for feature in range(sequence.shape[2]):

            f = interp1d(
                old_idx,
                sequence[:, landmark, feature],
                kind="linear",
                fill_value="extrapolate"
            )

            out[:, landmark, feature] = f(new_idx)
    return out

In [ ]:
def process_landmarks_folder(landmarks_root, output_root, fps_lookup, target_fps, target_frames):
    for root, _, files in os.walk(landmarks_root):
        for file in files:
            if not file.endswith(".npy"):
                continue

            in_path = os.path.join(root, file)
            rel_path = os.path.relpath(in_path, landmarks_root)
            out_path = os.path.join(output_root, rel_path)

            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            landmarks = np.load(in_path)

            video_name = rel_path.replace(".npy", ".mp4")
            fps = fps_lookup.get(video_name)

            if fps is None:
                print(f"FPS not found for {video_name}")
                continue

            if fps >= target_fps:
                landmarks = downsample(landmarks, target_fps, fps)

            landmarks = interpolate(landmarks, target_frames)

            np.save(out_path, landmarks)
    print("Done:", output_root)

In [ ]:
def landmarks_main(landmarks_root, output_root, report_path, target_fps, target_frames):
    report_df = pd.read_csv(report_path)
    fps_lookup = dict(zip(report_df["video"], report_df["fps"]))

    max_threads = max(1, os.cpu_count() or 1)
    ThreadPoolExecutor(max_threads).submit(
         process_landmarks_folder,
         landmarks_root, output_root, fps_lookup, target_fps, target_frames
    )

    #process_landmarks_folder(landmarks_root, output_root, fps_lookup, target_fps, target_frames)

In [ ]:
landmarks_main(LANDMARK_PATH, PROCESSED_LANDMARK_PATH, REPORT_PATH, TARGET_FPS, TARGET_FRAMES)